# Inter-Rater Reliability (IRR)

This notebook evaluates inter-annotator agreement on the BayFlood inspection
set. We measure how consistently three human annotators labeled the same
images as *Flooded* or *Not Flooded*.

> **Canonical source.** This notebook is the authoritative source for the
> inter-rater reliability numbers reported in the paper (Methods): pairwise
> Cohen's κ (annotator 1 vs 2 = 0.84, annotator 1 vs 3 = 0.96) and Fleiss'
> κ across all three annotators = 0.88. All three annotators are **human**;
> agreement is computed against the human `gt` label, never the VLM
> prediction. The older `0_irr_checks.ipynb` (reweighted/bootstrap variant)
> and `for_revisions/09_interrater_agreement.ipynb` are corrected two-annotator
> views that defer to this notebook for the reported values.

## 1. Setup and data loading

In [1]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, accuracy_score

In [2]:
# Load original inspection set and the two revision IRR samples.
# All three files contain *human* annotations.
inspection_set_annotated = pd.read_csv(
    "../../data/revisions/irr/bayflood_annotator1.csv"
)
annotator_2 = pd.read_csv("../../data/revisions/irr/bayflood_annotator2.csv")
annotator_3 = pd.read_csv("../../data/revisions/irr/bayflood_annotator3.csv")

# Drop the VLM prediction column so it cannot leak into any IRR computation.
# `gt` is the human label; `pred` is the VLM prediction and is not used here.
inspection_set_annotated = inspection_set_annotated.drop(columns=["pred"], errors="ignore")

In [3]:
inspection_set_annotated.head()

,image,gt
0,nlbx_0321cea2aba1e0031aad262982fd6bee.jpg,1
1,nlbx_9349f44a4398d9e694685945158ff79d.jpg,1
2,nlbx_3b21011a96dca4ac5579c558230f1fa8.jpg,0
3,nlbx_2a5121429509607e63c76bcdb9aa3f27.jpg,1
4,nlbx_877f4b3f318c142ed74aca3a7e364966.jpg,1


## 2. Build a common `frame_id` key

Each annotation source stores the image path slightly differently. We strip
the path prefix and `.jpg` extension to produce a `frame_id` that can be
joined across the three annotators.

In [4]:
inspection_set_annotated['frame_id'] = (
    inspection_set_annotated['image']
    .str.replace('/data/local-files/?d=/share/XXXX-19/nexar_data/training_datasets/street_flooding/all_no_letterboxing/', '', regex=False)
    .str.replace('.jpg', '', regex=False)
)

annotator_2['frame_id'] = (
    annotator_2['image']
    .str.replace('/data/local-files/?d=/share/ju/nexar_data/training_datasets/street_flooding/all_no_letterboxing/', '', regex=False)
    .str.replace('.jpg', '', regex=False)
)
annotator_3['frame_id'] = (
    annotator_3['image']
    .str.replace('/data/local-files/?d=/share/ju/nexar_data/training_datasets/street_flooding/all_no_letterboxing/', '', regex=False)
    .str.replace('.jpg', '', regex=False)
)

## 3. Preprocess the three human annotators

- **Annotator 1** — original paper labels. `label_irr1` is taken directly from
  the `gt` column (the integer-coded human choice).
- **Annotators 2 & 3** — hired for the revision. The first 100 rows of each
  file are a shared calibration/training batch and are excluded from the IRR
  analysis. Their free-text `choice` field is mapped to a binary label.

In [5]:
# Annotator 1: original paper annotations (human `gt`, not VLM `pred`).
annotator_1 = inspection_set_annotated.copy()
annotator_1['label_irr1'] = annotator_1['gt']

In [6]:
# Annotator 2: drop the 100-row calibration block, keep only valid labels.
annotator_2 = annotator_2.iloc[100:, :]
annotator_2['choice'] = annotator_2['choice'].astype(str).str.strip()
annotator_2 = annotator_2[annotator_2['choice'].isin(["Flooded", "Not Flooded"])]
annotator_2['label_irr2'] = annotator_2['choice'].map({"Flooded": 1, "Not Flooded": 0})

In [7]:
# Annotator 3: same preprocessing as Annotator 2.
annotator_3 = annotator_3.iloc[100:, :]
annotator_3['choice'] = annotator_3['choice'].astype(str).str.strip()
annotator_3 = annotator_3[annotator_3['choice'].isin(["Flooded", "Not Flooded"])]
annotator_3['label_irr3'] = annotator_3['choice'].map({"Flooded": 1, "Not Flooded": 0})

## 4. Merge annotators on the shared image set

We restrict the analysis to images labeled by all three annotators so that
every agreement statistic is computed on the same underlying sample.

In [8]:
merged_df = pd.merge(
    annotator_2[['frame_id', 'label_irr2']],
    annotator_1[['frame_id', 'label_irr1']],
    on='frame_id',
    how='inner',
)
merged_df = pd.merge(
    merged_df,
    annotator_3[['frame_id', 'label_irr3']],
    on='frame_id',
    how='inner',
)

merged_df['label_irr1'] = merged_df['label_irr1'].astype(int)
merged_df['label_irr2'] = merged_df['label_irr2'].astype(int)
merged_df['label_irr3'] = merged_df['label_irr3'].astype(int)

merged_df

,frame_id,label_irr2,label_irr1,label_irr3
0,nlbx_0320f09fa6e540a1d4718939d8082b1c,0,0,0
1,nlbx_163500fcb28d75868548e541a8869ce1,0,0,0
2,nlbx_e4cee750fb45528a166bf611ddd75556,0,0,0
3,nlbx_2fdb74c426dd5cf84a1a45d4e3b8e986,0,0,0
4,nlbx_4b576bcc8a05cfee35885e4229b4824e,0,0,0
...,...,...,...,...
393,nlbx_bf79e18a1ed7255008eb87ca60721b60,1,1,1
394,nlbx_886e2cc1959d9d43d19596cfd72dfc37,0,0,0
395,nlbx_fd91667a7686c8473ad191400481f838,1,1,1
396,nlbx_e78771f5a2c142ce168c88c806e52695,1,1,1


### Class balance per annotator

In [9]:
class_counts = pd.DataFrame(
    {
        'Flooded': [merged_df[col].sum() for col in ['label_irr1', 'label_irr2', 'label_irr3']],
        'Not Flooded': [len(merged_df) - merged_df[col].sum() for col in ['label_irr1', 'label_irr2', 'label_irr3']],
    },
    index=['Annotator 1', 'Annotator 2', 'Annotator 3'],
)
class_counts

,Flooded,Not Flooded
Annotator 1,169,229
Annotator 2,180,218
Annotator 3,167,231


## 5. Fleiss' kappa (three-way agreement)

Fleiss' kappa summarizes agreement across all three human annotators
simultaneously, correcting for chance.

In [10]:
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

label_matrix = merged_df[['label_irr1', 'label_irr2', 'label_irr3']]
fleiss_data, n_cat = aggregate_raters(label_matrix)
fleiss_kappa_score = fleiss_kappa(fleiss_data)
print(f"Fleiss' kappa (3 human annotators): {fleiss_kappa_score:.4f}")

Fleiss' kappa (3 human annotators): 0.8840


## 6. Pairwise Cohen's kappa and accuracy

Cohen's kappa is computed for each pair of human annotators on the shared
image set. Accuracy is the raw fraction of images on which the pair agrees
(useful as an interpretable companion to kappa).

In [11]:
pairs = [
    ("Annotator 1", "Annotator 2", 'label_irr1', 'label_irr2'),
    ("Annotator 1", "Annotator 3", 'label_irr1', 'label_irr3'),
    ("Annotator 2", "Annotator 3", 'label_irr2', 'label_irr3'),
]

rows = []
for name_a, name_b, col_a, col_b in pairs:
    rows.append({
        'Pair': f"{name_a} vs {name_b}",
        'N': len(merged_df),
        "Cohen's kappa": cohen_kappa_score(merged_df[col_a], merged_df[col_b]),
        'Accuracy': accuracy_score(merged_df[col_a], merged_df[col_b]),
    })

pairwise_kappa_df = pd.DataFrame(rows).set_index('Pair')
pairwise_kappa_df.round(4)

,N,Cohen's kappa,Accuracy
Pair,,,
Annotator 1 vs Annotator 2,398,0.8419,0.9221
Annotator 1 vs Annotator 3,398,0.9588,0.9799
Annotator 2 vs Annotator 3,398,0.8520,0.9271


## 7. Pairwise conditional probabilities

For each ordered pair of human annotators `(A, B)`, we report
`P(B = Flooded | A = Flooded)` and `P(B = Not Flooded | A = Not Flooded)`.
These conditional probabilities are more informative than overall accuracy
when the classes are imbalanced: they show how often one annotator
confirms the other's positive or negative call.

In [12]:
directed_pairs = [
    ("Annotator 1", "Annotator 2", 'label_irr1', 'label_irr2'),
    ("Annotator 2", "Annotator 1", 'label_irr2', 'label_irr1'),
    ("Annotator 1", "Annotator 3", 'label_irr1', 'label_irr3'),
    ("Annotator 3", "Annotator 1", 'label_irr3', 'label_irr1'),
    ("Annotator 2", "Annotator 3", 'label_irr2', 'label_irr3'),
    ("Annotator 3", "Annotator 2", 'label_irr3', 'label_irr2'),
]

rows = []
for name_a, name_b, col_a, col_b in directed_pairs:
    ct = pd.crosstab(merged_df[col_a], merged_df[col_b])
    # Guard against a class being absent for one annotator on this sample.
    p_flood = ct.loc[1, 1] / ct.loc[1].sum() if 1 in ct.index and 1 in ct.columns else float('nan')
    p_dry = ct.loc[0, 0] / ct.loc[0].sum() if 0 in ct.index and 0 in ct.columns else float('nan')
    rows.append({
        'Condition (A)': name_a,
        'Other (B)': name_b,
        'P(B=Flooded | A=Flooded)': p_flood,
        'P(B=Not Flooded | A=Not Flooded)': p_dry,
    })

pairwise_cond_df = pd.DataFrame(rows)
pairwise_cond_df.round(4)

,Condition (A),Other (B),P(B=Flooded | A=Flooded),P(B=Not Flooded | A=Not Flooded)
0,Annotator 1,Annotator 2,0.9408,0.9083
1,Annotator 2,Annotator 1,0.8833,0.9541
2,Annotator 1,Annotator 3,0.9704,0.9869
3,Annotator 3,Annotator 1,0.9820,0.9784
4,Annotator 2,Annotator 3,0.8833,0.9633
5,Annotator 3,Annotator 2,0.9521,0.9091


### Pairwise confusion matrices

For reference, the raw 2x2 confusion matrices underlying the conditional
probabilities above.

In [13]:
for name_a, name_b, col_a, col_b in [
    ("Annotator 1", "Annotator 2", 'label_irr1', 'label_irr2'),
    ("Annotator 1", "Annotator 3", 'label_irr1', 'label_irr3'),
    ("Annotator 2", "Annotator 3", 'label_irr2', 'label_irr3'),
]:
    print(f"{name_a} (rows) vs {name_b} (cols)")
    print(pd.crosstab(merged_df[col_a], merged_df[col_b], rownames=[name_a], colnames=[name_b]))
    print()

Annotator 1 (rows) vs Annotator 2 (cols)
Annotator 2    0    1
Annotator 1          
0            208   21
1             10  159

Annotator 1 (rows) vs Annotator 3 (cols)
Annotator 3    0    1
Annotator 1          
0            226    3
1              5  164

Annotator 2 (rows) vs Annotator 3 (cols)
Annotator 3    0    1
Annotator 2          
0            210    8
1             21  159



## 8. Coverage check: images missing from at least one annotator

Sanity check that the IRR sample is nearly fully covered by all three human
annotators. Any frame that is present for only a subset is listed below.

In [14]:
only_annotations_irr1 = pd.read_csv('../../data/revisions/irr/inspection_set_IRR.csv')
only_annotations_irr1['frame_id'] = (
    only_annotations_irr1['image']
    .str.replace('/data/local-files/?d=/share/ju/nexar_data/training_datasets/street_flooding/all_no_letterboxing/', '', regex=False)
    .str.replace('.jpg', '', regex=False)
)

annotator_1_irr = annotator_1[annotator_1['frame_id'].isin(only_annotations_irr1['frame_id'])]
ids_1 = set(annotator_1_irr['frame_id'])
ids_2 = set(annotator_2['frame_id'])
ids_3 = set(annotator_3['frame_id'])

all_ids = ids_1 | ids_2 | ids_3
common_ids = ids_1 & ids_2 & ids_3
not_in_all = all_ids - common_ids

print(f"Total unique frame_ids across all annotators: {len(all_ids)}")
print(f"frame_ids present in all three: {len(common_ids)}")
print(f"frame_ids NOT present in all three: {len(not_in_all)}")

missing_df = pd.DataFrame([
    {
        'frame_id': fid,
        'in_annotator1': fid in ids_1,
        'in_annotator2': fid in ids_2,
        'in_annotator3': fid in ids_3,
    }
    for fid in sorted(not_in_all)
])
missing_df

Total unique frame_ids across all annotators: 502
frame_ids present in all three: 398
frame_ids NOT present in all three: 104


,frame_id,in_annotator1,in_annotator2,in_annotator3
0,nlbx_072704874c920879a1b8281d3699ee2e,True,False,False
1,nlbx_076d92c3815d9608fc80e0476f00df2b,True,False,False
2,nlbx_0b1a8e5ed67029afe8fc53368f4c32db,True,False,False
3,nlbx_0b3f34b519117d4a10f8f81f1c79acc7,True,False,False
4,nlbx_0bab426861579e03fc59508b39b797eb,True,False,False
...,...,...,...,...
99,nlbx_f3e598b567976e75899f5b99db8c70dc,True,False,False
100,nlbx_f81b40265b056e46fa82094a6e9250e7,True,False,False
101,nlbx_f8d79fe059864bf4dc948ea58ad5f428,True,False,False
102,nlbx_fc0dd02fa4eaa78ccbd5379c8464d0a6,True,False,False
